# Lesson 03 - Agentic Design Patterns

V tejto lekcii preskúmame tri základné návrhové vzory pre vytváranie efektívnych AI agentov:

1. **Jasné pokyny pre agenta** — Tvorba presných podnetov definujúcich rolu, ktoré usmerňujú správanie agenta  
2. **Štruktúrovaný výstup pomocou Pydantic modelov** — Zabezpečenie, že agenti vracajú predvídateľné, overené dáta  
3. **Agent s jednou zodpovednosťou** — Navrhovanie zameraných agentov, ktorí robia jednu vec dobre  

Každý vzor použijeme na scenár **odporúčateľa cestovných destinácií**, postupne budujeme systém, ktorý môže navrhovať destinácie, kontrolovať dostupnosť a riešiť logistiku.


## Nastavenie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Vzor 1: Jasné pokyny pre agenta

Najvplyvnejší vzor je zároveň ten najjednoduchší: napísať jasné, podrobné pokyny pre vášho agenta.

Dobré pokyny definujú:
- **Kto** agent je (osobnosť a tón)
- **Čo** by mal robiť (postupné zodpovednosti)
- **Ako** by sa mal správať (obmedzenia a štýl)

Nižšie vytvárame cestovného konzultanta s explicitnými pokynmi, ktoré formujú každú jeho odpoveď.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Vzor 2: Štruktúrovaný výstup s modelmi Pydantic

Voľný text je užitočný pre konverzácie, ale nasledujúce systémy potrebujú štruktúrované dáta.  
Spojením **modelov Pydantic** s **nástrojovou funkciou** môžeme:

- Definovať presnú schému pre výstup agenta  
- Automaticky overovať odpovede  
- Spoľahlivo integrovať výsledky agenta do logiky aplikácie  

Tiež predstavujeme nástroj, ktorý vracia údaje o cieľovej destinácii, aby agent zakotvil svoje odporúčania v skutočných dátach.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Vzor 3: Agenti s Jednou Zodpovednosťou

Zložité úlohy profitujú zo rozdelenia práce medzi viacerých zameraných agentov, z ktorých každý má jednu zodpovednosť:

- **Odborník na destinácie**, ktorý pozná miesta a dostupnosť
- **Plánovač logistiky**, ktorý rieši lety, hotely a itineráre

Toto odráža princíp softvérového inžinierstva *oddelenia zodpovedností* — každý agent je jednoduchší na testovanie, údržbu a samostatné zlepšovanie.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Zhrnutie

V tejto lekcii sme aplikovali tri agentné návrhové vzory na scenár odporúčania cestovania:

| Vzor | Hlavná myšlienka | Výhoda |
|---|---|---|
| **Jasné pokyny** | Definovať postavu, zodpovednosti a obmedzenia už na začiatku | Konzistentné správanie agenta v súlade so značkou |
| **Štruktúrovaný výstup** | Použiť Pydantic modely ako formát odpovede | Validované, strojovo čitateľné výsledky |
| **Jedna zodpovednosť** | Každému agentovi prideliť jednu zameranú úlohu | Jednoduchšie testovanie, údržba a skladanie |

Tieto vzory sa prirodzene kombinujú — môžete spojiť jasné pokyny so štruktúrovaným výstupom v agente s jednou zodpovednosťou na vytvorenie robustných systémov pripravených na produkciu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
